# Interactive Visualization of ALMA Archive Relationships

This notebook presents the structural findings from
`02_archive_data_relationships.ipynb` through interactive visualizations.

The selected Member OUS datasets illustrate simple, multi-source,
multi-ASDM, and mosaic data structures. They are structurally informative
examples and are not intended to represent Archive-wide proportions.

In [8]:
import re
import sys

import ipywidgets
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import pyvo

from IPython.display import clear_output, display


pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)


print("Python executable:", sys.executable)
print("Plotly version:", plotly.__version__)
print("ipywidgets version:", ipywidgets.__version__)

Python executable: /Users/nana/opt/anaconda3/envs/alma-duplication/bin/python
Plotly version: 6.9.0
ipywidgets version: 8.1.9


## 1. Representative Member OUS Datasets

Six Member OUS datasets identified in Notebook 2 are used to illustrate
different Archive structures.

The examples include single-source, multi-source, multi-ASDM, large-grid,
and mosaic cases. They were selected for structural interpretation and are
not statistically representative of the complete Archive.

In [9]:
TAP_URL = "https://almascience.eso.org/tap"

tap_service = pyvo.dal.TAPService(TAP_URL)


def run_tap_query(query, maxrec):
    """Run an ALMA TAP query and return a Pandas DataFrame."""

    result = tap_service.search(
        query,
        maxrec=maxrec,
    )

    return result.to_table().to_pandas()


print("TAP service ready:", TAP_URL)

TAP service ready: https://almascience.eso.org/tap


In [10]:
representative_members = {
    "Simple — 1 source × 4 SPWs":
        "uid://A001/X15a1/Xe3b",

    "Multi-ASDM — 2 sources × 4 SPWs":
        "uid://A001/X3833/X1022",

    "Large source grid — 36 sources × 4 SPWs":
        "uid://A001/X3819/X186",

    "Large spectral grid — 16 sources × 16 SPWs":
        "uid://A001/X3788/Xc584",

    "Mosaic — 8 sources × 14 SPWs":
        "uid://A001/X3833/X4d0",

    "Mixed mosaic-state case":
        "uid://A001/X383d/X43a",
}


member_uid_sql = ",\n    ".join(
    f"'{member_uid}'"
    for member_uid in representative_members.values()
)


uid_to_case = {
    member_uid: case_name
    for case_name, member_uid
    in representative_members.items()
}


display(
    pd.DataFrame(
        {
            "case": representative_members.keys(),
            "member_ous_uid": representative_members.values(),
        }
    )
)

,case,member_ous_uid
0,Simple — 1 source × 4 SPWs,uid://A001/X15a1/Xe3b
1,Multi-ASDM — 2 sources × 4 SPWs,uid://A001/X3833/X1022
2,Large source grid — 36 sources × 4 SPWs,uid://A001/X3819/X186
3,Large spectral grid — 16 sources × 16 SPWs,uid://A001/X3788/Xc584
4,Mosaic — 8 sources × 14 SPWs,uid://A001/X3833/X4d0
5,Mixed mosaic-state case,uid://A001/X383d/X43a


In [11]:
count_query = f"""
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid IN (
    {member_uid_sql}
)
"""


count_df = run_tap_query(
    count_query,
    maxrec=1,
)


expected_rows = int(
    count_df.loc[0, "total_rows"]
)


print("Expected Archive rows:", expected_rows)

Expected Archive rows: 532


In [12]:
representative_query = f"""
SELECT
    proposal_id,
    group_ous_uid,
    member_ous_uid,
    asdm_uid,
    obs_id,
    target_name,
    s_ra,
    s_dec,
    s_region,
    frequency,
    bandwidth,
    frequency_support,
    spatial_resolution,
    sensitivity_10kms,
    cont_sensitivity_bandwidth,
    antenna_arrays,
    is_mosaic,
    t_min,
    t_max
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid IN (
    {member_uid_sql}
)
"""


archive_visual_df = run_tap_query(
    representative_query,
    maxrec=max(expected_rows, 1),
)


print("Expected rows:", expected_rows)
print("Retrieved rows:", len(archive_visual_df))
print(
    "Complete result:",
    len(archive_visual_df) == expected_rows,
)


archive_visual_df.head()

Expected rows: 532
Retrieved rows: 532
Complete result: True


,proposal_id,group_ous_uid,member_ous_uid,asdm_uid,obs_id,target_name,s_ra,s_dec,s_region,frequency,bandwidth,frequency_support,spatial_resolution,sensitivity_10kms,cont_sensitivity_bandwidth,antenna_arrays,is_mosaic,t_min,t_max
0,2021.1.01123.L,uid://A001/X15a1/Xe36,uid://A001/X15a1/Xe3b,uid://A002/Xf1948b/X6406,uid://A001/X15a1/Xe3b.source.HD34282.spw.16,HD34282,79.001988,-9.809838,Polygon ICRS 79.005021 -9.812644 79.003759 -9.813547 79.002249 -9.813929 79.000702 -9.813737 78.999759 -9.813299 78....,330.610912,6.250000e+07,"[330.58..330.64GHz,30.52kHz,11.4mJy/beam@10km/s,4.8mJy/beam@native, XX YY] U [330.59..332.59GHz,976.56kHz,10.2mJy/be...",3.670041,11.397958,0.720991,J501:CM10 J502:CM02 J503:CM03 J504:CM12 J505:CM08 J506:CM05 N603:CM09 N604:CM11,F,59498.294193,59502.411376
1,2021.1.01123.L,uid://A001/X15a1/Xe36,uid://A001/X15a1/Xe3b,uid://A002/Xf1948b/X6406,uid://A001/X15a1/Xe3b.source.HD34282.spw.18,HD34282,79.001988,-9.809838,Polygon ICRS 79.005021 -9.812644 79.003759 -9.813547 79.002249 -9.813929 79.000702 -9.813737 78.999759 -9.813299 78....,331.592941,2.000000e+09,"[330.58..330.64GHz,30.52kHz,11.4mJy/beam@10km/s,4.8mJy/beam@native, XX YY] U [330.59..332.59GHz,976.56kHz,10.2mJy/be...",3.670041,10.164604,0.720991,J501:CM10 J502:CM02 J503:CM03 J504:CM12 J505:CM08 J506:CM05 N603:CM09 N604:CM11,F,59498.294193,59502.411376
2,2021.1.01123.L,uid://A001/X15a1/Xe36,uid://A001/X15a1/Xe3b,uid://A002/Xf1948b/X6406,uid://A001/X15a1/Xe3b.source.HD34282.spw.20,HD34282,79.001988,-9.809838,Polygon ICRS 79.005021 -9.812644 79.003759 -9.813547 79.002249 -9.813929 79.000702 -9.813737 78.999759 -9.813299 78....,342.906620,6.250000e+07,"[330.58..330.64GHz,30.52kHz,11.4mJy/beam@10km/s,4.8mJy/beam@native, XX YY] U [330.59..332.59GHz,976.56kHz,10.2mJy/be...",3.670041,8.526328,0.720991,J501:CM10 J502:CM02 J503:CM03 J504:CM12 J505:CM08 J506:CM05 N603:CM09 N604:CM11,F,59498.294193,59502.411376
3,2021.1.01123.L,uid://A001/X15a1/Xe36,uid://A001/X15a1/Xe3b,uid://A002/Xf1948b/X6406,uid://A001/X15a1/Xe3b.source.HD34282.spw.22,HD34282,79.001988,-9.809838,Polygon ICRS 79.005021 -9.812644 79.003759 -9.813547 79.002249 -9.813929 79.000702 -9.813737 78.999759 -9.813299 78....,345.819889,6.250000e+07,"[330.58..330.64GHz,30.52kHz,11.4mJy/beam@10km/s,4.8mJy/beam@native, XX YY] U [330.59..332.59GHz,976.56kHz,10.2mJy/be...",3.670041,9.941859,0.720991,J501:CM10 J502:CM02 J503:CM03 J504:CM12 J505:CM08 J506:CM05 N603:CM09 N604:CM11,F,59498.294193,59502.411376
4,2025.1.01547.S,uid://A001/X3833/X4cf,uid://A001/X3833/X4d0,uid://A002/X133dde2/X2a69e,uid://A001/X3833/X4d0.source.SDC24.433.spw.34,SDC24.433,279.169134,-7.666242,Polygon ICRS 279.174453 -7.712487 279.165193 -7.713646 279.157685 -7.709855 279.153039 -7.700806 279.145279 -7.69530...,86.740478,5.859375e+07,"[85.20..85.26GHz,141.11kHz,98.2mJy/beam@10km/s,21.6mJy/beam@native, XX YY] U [85.41..85.47GHz,141.11kHz,98.1mJy/beam...",11.285004,97.341720,5.523380,J501:CM10 J502:CM03 J503:CM04 J504:CM12 J505:CM08 J506:CM05 N601:CM07 N602:CM01 N603:CM09 N604:CM11 N605:CM02,T,61035.742270,61237.233194


## 2. Source–SPW Row Structure

In the tested records, an `obs_id` contains a source component and a
spectral-window identifier.

Each cell in the following visualization represents one Archive row.
The visualization tests whether the records within a Member OUS form a
complete source–SPW grid.

In [13]:
obs_id_pattern = re.compile(
    r"\.source\.(?P<obs_id_source>.+)"
    r"\.spw\.(?P<spw_identifier>[^.]+)$"
)


parsed_obs_id = (
    archive_visual_df["obs_id"]
    .astype(str)
    .str.extract(obs_id_pattern)
)


visual_df = pd.concat(
    [
        archive_visual_df.reset_index(drop=True),
        parsed_obs_id.reset_index(drop=True),
    ],
    axis=1,
)


visual_df["spw_numeric"] = pd.to_numeric(
    visual_df["spw_identifier"],
    errors="coerce",
)


visual_df["frequency"] = pd.to_numeric(
    visual_df["frequency"],
    errors="coerce",
)


visual_df["bandwidth_ghz"] = (
    pd.to_numeric(
        visual_df["bandwidth"],
        errors="coerce",
    )
    / 1e9
)


visual_df["case"] = visual_df[
    "member_ous_uid"
].map(uid_to_case)


print("Rows:", len(visual_df))
print(
    "Unparsed obs_id values:",
    visual_df["obs_id_source"].isna().sum(),
)

Rows: 532
Unparsed obs_id values: 0


In [14]:
member_structure_df = (
    visual_df
    .groupby("member_ous_uid", dropna=False)
    .agg(
        archive_rows=("obs_id", "size"),
        source_count=("obs_id_source", "nunique"),
        spw_count=("spw_identifier", "nunique"),
        asdm_count=("asdm_uid", "nunique"),
        coordinate_count=("s_ra", "nunique"),
        footprint_count=("s_region", "nunique"),
        mosaic_state_count=("is_mosaic", "nunique"),
    )
    .reset_index()
)


member_structure_df["expected_grid_rows"] = (
    member_structure_df["source_count"]
    * member_structure_df["spw_count"]
)


member_structure_df["complete_grid"] = (
    member_structure_df["archive_rows"]
    == member_structure_df["expected_grid_rows"]
)


member_structure_df["case"] = (
    member_structure_df["member_ous_uid"]
    .map(uid_to_case)
)


member_structure_df[
    [
        "case",
        "archive_rows",
        "source_count",
        "spw_count",
        "expected_grid_rows",
        "asdm_count",
        "coordinate_count",
        "footprint_count",
        "mosaic_state_count",
        "complete_grid",
    ]
]

,case,archive_rows,source_count,spw_count,expected_grid_rows,asdm_count,coordinate_count,footprint_count,mosaic_state_count,complete_grid
0,Simple — 1 source × 4 SPWs,4,1,4,4,1,1,1,1,True
1,Large spectral grid — 16 sources × 16 SPWs,256,16,16,256,1,16,16,1,True
2,Large source grid — 36 sources × 4 SPWs,144,36,4,144,1,36,36,1,True
3,Multi-ASDM — 2 sources × 4 SPWs,8,2,4,8,2,2,2,1,True
4,Mosaic — 8 sources × 14 SPWs,112,8,14,112,1,8,8,1,True
5,Mixed mosaic-state case,8,2,4,8,1,2,2,2,True


In [15]:
mean_spw_frequency = (
    visual_df
    .groupby(
        [
            "member_ous_uid",
            "spw_identifier",
        ],
        dropna=False,
    )["frequency"]
    .transform("mean")
)


visual_df["frequency_offset_mhz"] = (
    visual_df["frequency"]
    - mean_spw_frequency
) * 1000

In [16]:
def build_source_spw_matrix(member_uid):
    """Create an interactive Source × SPW row matrix."""

    member_df = visual_df.loc[
        visual_df["member_ous_uid"] == member_uid
    ].copy()

    spw_order = (
        member_df[
            [
                "spw_identifier",
                "spw_numeric",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "spw_numeric",
                "spw_identifier",
            ],
            na_position="last",
        )["spw_identifier"]
        .astype(str)
        .tolist()
    )

    source_order = sorted(
        member_df["obs_id_source"]
        .astype(str)
        .unique()
        .tolist()
    )

    figure = px.scatter(
        member_df,
        x="spw_identifier",
        y="obs_id_source",
        color="frequency_offset_mhz",
        hover_name="obs_id",
        hover_data={
            "target_name": True,
            "asdm_uid": True,
            "frequency": ":.9f",
            "frequency_offset_mhz": ":.4f",
            "bandwidth_ghz": ":.6f",
            "sensitivity_10kms": ":.3f",
            "is_mosaic": True,
            "spw_numeric": False,
        },
        category_orders={
            "spw_identifier": spw_order,
            "obs_id_source": source_order,
        },
        color_continuous_scale="RdBu_r",
        color_continuous_midpoint=0,
        labels={
            "spw_identifier": "Logical SPW identifier",
            "obs_id_source": "Source context",
            "frequency_offset_mhz":
                "Frequency offset (MHz)",
        },
        title=uid_to_case.get(
            member_uid,
            member_uid,
        ),
    )

    figure.update_traces(
        marker={
            "symbol": "square",
            "size": 18,
            "line": {
                "width": 0.5,
                "color": "gray",
            },
        }
    )

    figure.update_layout(
        width=1050,
        height=max(
            460,
            28 * len(source_order) + 190,
        ),
        coloraxis_colorbar={
            "title": "Offset<br>(MHz)",
        },
        margin={
            "l": 90,
            "r": 40,
            "t": 80,
            "b": 70,
        },
    )

    figure.update_yaxes(
        autorange="reversed",
    )

    return figure

In [17]:
member_dropdown = ipywidgets.Dropdown(
    options=[
        (case_name, member_uid)
        for case_name, member_uid
        in representative_members.items()
    ],
    value=next(
        iter(representative_members.values())
    ),
    description="Member:",
    layout=ipywidgets.Layout(
        width="720px",
    ),
    style={
        "description_width": "80px",
    },
)


matrix_output = ipywidgets.Output()


def update_matrix(change=None):
    member_uid = member_dropdown.value

    member_df = visual_df.loc[
        visual_df["member_ous_uid"] == member_uid
    ]

    with matrix_output:
        clear_output(wait=True)

        print(
            f"{member_df['obs_id_source'].nunique()} sources × "
            f"{member_df['spw_identifier'].nunique()} SPWs = "
            f"{len(member_df)} Archive rows"
        )

        print(
            "ASDMs:",
            member_df["asdm_uid"].nunique(),
            "| Coordinates:",
            member_df["s_ra"].nunique(),
            "| Footprints:",
            member_df["s_region"].nunique(),
        )

        display(
            build_source_spw_matrix(member_uid)
        )


member_dropdown.observe(
    update_matrix,
    names="value",
)


display(
    member_dropdown,
    matrix_output,
)


update_matrix()

Dropdown(description='Member:', layout=Layout(width='720px'), options=(('Simple — 1 source × 4 SPWs', 'uid://A…

Output()

## 3. Interactive Spectral Fingerprint

The upper visualization shows the approximate row-level frequency interval
associated with every source–SPW record.

The preliminary interval is calculated as:

$$
\nu_{\mathrm{low}}
=
\nu_{\mathrm{center}}
-
\frac{\mathrm{bandwidth}}{2}
$$

$$
\nu_{\mathrm{high}}
=
\nu_{\mathrm{center}}
+
\frac{\mathrm{bandwidth}}{2}
$$

The lower visualization magnifies one selected SPW and shows the
source-dependent frequency offsets in MHz.

These row-level intervals are exploratory approximations. They must not replace
the parsed `frequency_support` intervals, which will be investigated in
Notebook 3.

In [18]:
visual_df["frequency_low_ghz"] = (
    visual_df["frequency"]
    - visual_df["bandwidth_ghz"] / 2
)


visual_df["frequency_high_ghz"] = (
    visual_df["frequency"]
    + visual_df["bandwidth_ghz"] / 2
)


visual_df[
    [
        "member_ous_uid",
        "obs_id_source",
        "spw_identifier",
        "frequency_low_ghz",
        "frequency",
        "frequency_high_ghz",
        "bandwidth_ghz",
    ]
].head()

,member_ous_uid,obs_id_source,spw_identifier,frequency_low_ghz,frequency,frequency_high_ghz,bandwidth_ghz
0,uid://A001/X15a1/Xe3b,HD34282,16,330.579662,330.610912,330.642162,0.062500
1,uid://A001/X15a1/Xe3b,HD34282,18,330.592941,331.592941,332.592941,2.000000
2,uid://A001/X15a1/Xe3b,HD34282,20,342.875370,342.906620,342.937870,0.062500
3,uid://A001/X15a1/Xe3b,HD34282,22,345.788639,345.819889,345.851139,0.062500
4,uid://A001/X3833/X4d0,SDC24.433,34,86.711182,86.740478,86.769775,0.058594


In [19]:
invalid_frequency_intervals = visual_df.loc[
    visual_df["frequency_low_ghz"].isna()
    | visual_df["frequency_high_ghz"].isna()
    | (
        visual_df["frequency_low_ghz"]
        >= visual_df["frequency_high_ghz"]
    )
]


print(
    "Invalid frequency intervals:",
    len(invalid_frequency_intervals),
)

Invalid frequency intervals: 0


In [20]:
def get_ordered_spws(member_df):
    """Return SPW identifiers in numeric order where possible."""

    return (
        member_df[
            [
                "spw_identifier",
                "spw_numeric",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "spw_numeric",
                "spw_identifier",
            ],
            na_position="last",
        )["spw_identifier"]
        .astype(str)
        .tolist()
    )

In [21]:
def build_frequency_coverage_figure(member_uid):
    """Show approximate frequency intervals for all source–SPW rows."""

    member_df = visual_df.loc[
        visual_df["member_ous_uid"] == member_uid
    ].copy()

    member_df = member_df.loc[
        member_df["frequency_low_ghz"].notna()
        & member_df["frequency_high_ghz"].notna()
    ].copy()

    spw_order = get_ordered_spws(member_df)

    source_order = sorted(
        member_df["obs_id_source"]
        .astype(str)
        .unique()
        .tolist()
    )

    colour_palette = px.colors.qualitative.Dark24

    spw_colour_map = {
        spw_identifier:
            colour_palette[
                index % len(colour_palette)
            ]
        for index, spw_identifier
        in enumerate(spw_order)
    }

    figure = go.Figure()

    for spw_identifier in spw_order:
        spw_df = member_df.loc[
            member_df["spw_identifier"].astype(str)
            == str(spw_identifier)
        ].copy()

        custom_data = spw_df[
            [
                "frequency",
                "frequency_low_ghz",
                "frequency_high_ghz",
                "bandwidth_ghz",
                "asdm_uid",
                "sensitivity_10kms",
                "is_mosaic",
                "obs_id",
            ]
        ].to_numpy()

        figure.add_trace(
            go.Bar(
                x=spw_df["bandwidth_ghz"],
                base=spw_df["frequency_low_ghz"],
                y=spw_df["obs_id_source"],
                orientation="h",
                width=0.62,
                name=f"SPW {spw_identifier}",
                marker_color=spw_colour_map[
                    spw_identifier
                ],
                customdata=custom_data,
                hovertemplate=(
                    "<b>%{y}</b><br>"
                    f"SPW: {spw_identifier}<br>"
                    "Center: %{customdata[0]:.9f} GHz<br>"
                    "Approx. low: %{customdata[1]:.9f} GHz<br>"
                    "Approx. high: %{customdata[2]:.9f} GHz<br>"
                    "Bandwidth: %{customdata[3]:.6f} GHz<br>"
                    "ASDM: %{customdata[4]}<br>"
                    "Sensitivity 10 km/s: "
                    "%{customdata[5]} mJy/beam<br>"
                    "Mosaic: %{customdata[6]}<br>"
                    "obs_id: %{customdata[7]}"
                    "<extra></extra>"
                ),
            )
        )

    figure.update_layout(
        title=(
            "Approximate frequency coverage — "
            + uid_to_case.get(
                member_uid,
                member_uid,
            )
        ),
        xaxis_title="Observed sky frequency (GHz)",
        yaxis_title="Source context",
        barmode="overlay",
        width=1100,
        height=max(
            480,
            27 * len(source_order) + 200,
        ),
        legend_title="Logical SPW",
        hovermode="closest",
        margin={
            "l": 100,
            "r": 40,
            "t": 90,
            "b": 70,
        },
    )

    figure.update_yaxes(
        categoryorder="array",
        categoryarray=source_order,
        autorange="reversed",
    )

    return figure

In [24]:
def build_frequency_offset_figure(
    member_uid,
    spw_identifier,
):
    """Magnify source-dependent frequency offsets for one SPW."""

    spw_df = visual_df.loc[
        (
            visual_df["member_ous_uid"]
            == member_uid
        )
        & (
            visual_df["spw_identifier"].astype(str)
            == str(spw_identifier)
        )
    ].copy()

    spw_df["asdm_label"] = (
        spw_df["asdm_uid"]
        .fillna("Missing ASDM")
        .astype(str)
    )

    source_order = sorted(
        spw_df["obs_id_source"]
        .astype(str)
        .unique()
        .tolist()
    )

    figure = px.scatter(
        spw_df,
        x="frequency_offset_mhz",
        y="obs_id_source",
        color="asdm_label",
        symbol="is_mosaic",
        hover_name="obs_id",
        hover_data={
            "frequency": ":.9f",
            "frequency_offset_mhz": ":.6f",
            "bandwidth_ghz": ":.6f",
            "target_name": True,
            "sensitivity_10kms": ":.3f",
            "asdm_label": False,
        },
        category_orders={
            "obs_id_source": source_order,
        },
        labels={
            "frequency_offset_mhz":
                "Offset from Member–SPW mean (MHz)",
            "obs_id_source":
                "Source context",
            "asdm_label":
                "ASDM",
            "is_mosaic":
                "Mosaic",
        },
        title=(
            f"Source-dependent frequency offsets — "
            f"SPW {spw_identifier}"
        ),
    )

    figure.update_traces(
        marker={
            "size": 11,
            "line": {
                "width": 0.7,
                "color": "gray",
            },
        }
    )

    figure.add_vline(
        x=0,
        line_width=1,
        line_dash="dash",
        line_color="gray",
        annotation_text="SPW mean",
        annotation_position="top",
    )

    figure.update_layout(
        width=1050,
        height=max(
            430,
            27 * len(source_order) + 180,
        ),
        legend_title="Execution context",
        margin={
            "l": 100,
            "r": 40,
            "t": 90,
            "b": 70,
        },
    )

    figure.update_yaxes(
        categoryorder="array",
        categoryarray=source_order,
        autorange="reversed",
    )

    return figure

In [26]:
initial_member_uid = next(
    iter(representative_members.values())
)


initial_member_df = visual_df.loc[
    visual_df["member_ous_uid"]
    == initial_member_uid
]


initial_spws = get_ordered_spws(
    initial_member_df
)


spectral_member_dropdown = ipywidgets.Dropdown(
    options=[
        (case_name, member_uid)
        for case_name, member_uid
        in representative_members.items()
    ],
    value=initial_member_uid,
    description="Member:",
    layout=ipywidgets.Layout(
        width="720px",
    ),
    style={
        "description_width": "80px",
    },
)


spectral_spw_dropdown = ipywidgets.Dropdown(
    options=[
        (f"SPW {spw}", spw)
        for spw in initial_spws
    ],
    value=initial_spws[0],
    description="SPW:",
    layout=ipywidgets.Layout(
        width="350px",
    ),
    style={
        "description_width": "80px",
    },
)


spectral_output = ipywidgets.Output()


updating_spw_options = False

In [27]:
def on_spectral_member_change(change):
    """Refresh available SPWs after Member selection."""

    global updating_spw_options

    updating_spw_options = True

    member_uid = spectral_member_dropdown.value

    member_df = visual_df.loc[
        visual_df["member_ous_uid"]
        == member_uid
    ]

    available_spws = get_ordered_spws(
        member_df
    )

    spectral_spw_dropdown.options = [
        (f"SPW {spw}", spw)
        for spw in available_spws
    ]

    spectral_spw_dropdown.value = (
        available_spws[0]
    )

    updating_spw_options = False

    render_spectral_fingerprint()


def on_spectral_spw_change(change):
    """Refresh magnified view after SPW selection."""

    if not updating_spw_options:
        render_spectral_fingerprint()


spectral_member_dropdown.observe(
    on_spectral_member_change,
    names="value",
)


spectral_spw_dropdown.observe(
    on_spectral_spw_change,
    names="value",
)

In [30]:
def render_spectral_fingerprint():
    """Render linked full-coverage and fine-offset views."""

    member_uid = spectral_member_dropdown.value
    spw_identifier = spectral_spw_dropdown.value

    member_df = visual_df.loc[
        visual_df["member_ous_uid"]
        == member_uid
    ]

    selected_spw_df = member_df.loc[
        member_df["spw_identifier"].astype(str)
        == str(spw_identifier)
    ]

    maximum_offset = (
        selected_spw_df["frequency_offset_mhz"]
        .abs()
        .max()
    )

    with spectral_output:
        clear_output(wait=True)

        print(
            uid_to_case.get(
                member_uid,
                member_uid,
            )
        )

        print(
            f"{member_df['obs_id_source'].nunique()} sources | "
            f"{member_df['spw_identifier'].nunique()} SPWs | "
            f"{member_df['asdm_uid'].nunique()} ASDMs"
        )

        print(
            f"Selected SPW {spw_identifier}: "
            f"maximum absolute offset = "
            f"{maximum_offset:.6f} MHz"
        )

        display(
            build_frequency_coverage_figure(
                member_uid
            )
        )

        display(
            build_frequency_offset_figure(
                member_uid,
                spw_identifier,
            )
        )

In [31]:
display(
    ipywidgets.VBox(
        [
            spectral_member_dropdown,
            spectral_spw_dropdown,
            spectral_output,
        ]
    )
)


render_spectral_fingerprint()

In [32]:
import astropy.units as u

from astropy.coordinates import (
    SkyCoord,
    SkyOffsetFrame,
)

In [33]:
from astropy.coordinates import SkyCoord

print("Astropy coordinate tools ready")

Astropy coordinate tools ready


## 4. Interactive Sky Centres and Spatial Footprints

This visualization compares the Archive reference coordinates
(`s_ra`, `s_dec`) with the spatial coverage stored in `s_region`.

The display uses a local sky-offset frame centred on the selected Member OUS:

- north is up;
- east is to the left;
- both axes use the same angular scale;
- offsets are shown in arcseconds;
- centre markers and spatial footprints can be displayed independently.

The `s_region` field is an Archive spatial-coverage representation. It must not
automatically be interpreted as an exact list of mosaic pointings, a primary
beam, or a half-power coverage contour.

Similarly, a Polygon footprint does not by itself imply that the observation is
a mosaic. The footprint geometry and the `is_mosaic` flag are displayed as
separate properties.

In [34]:
stcs_number_pattern = re.compile(
    r"[-+]?"
    r"(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[Ee][-+]?\d+)?"
)


def parse_s_region(region_value):
    """Parse simple ALMA STC-S Polygon and Circle regions."""

    result = {
        "geometry_type": "Missing",
        "coordinate_frame": None,
        "vertices_deg": None,
        "centre_ra_deg": None,
        "centre_dec_deg": None,
        "radius_deg": None,
        "parse_status": "missing",
    }

    if region_value is None:
        return result

    if isinstance(region_value, bytes):
        region_text = region_value.decode(
            "utf-8",
            errors="replace",
        )
    else:
        region_text = str(region_value)

    region_text = region_text.strip()

    if not region_text:
        return result

    tokens = region_text.split()

    geometry_type = tokens[0].upper()

    coordinate_frame = (
        tokens[1].upper()
        if len(tokens) > 1
        else None
    )

    numbers = [
        float(number)
        for number in stcs_number_pattern.findall(
            region_text
        )
    ]

    result["geometry_type"] = geometry_type.title()
    result["coordinate_frame"] = coordinate_frame

    if coordinate_frame != "ICRS":
        result["parse_status"] = "unsupported_frame"
        return result

    if geometry_type == "POLYGON":
        if len(numbers) < 6 or len(numbers) % 2 != 0:
            result["parse_status"] = "invalid_polygon"
            return result

        vertices = np.asarray(
            numbers,
            dtype=float,
        ).reshape(-1, 2)

        result["vertices_deg"] = vertices
        result["parse_status"] = "parsed"

        return result

    if geometry_type == "CIRCLE":
        if len(numbers) != 3:
            result["parse_status"] = "invalid_circle"
            return result

        result["centre_ra_deg"] = numbers[0]
        result["centre_dec_deg"] = numbers[1]
        result["radius_deg"] = numbers[2]
        result["parse_status"] = "parsed"

        return result

    result["parse_status"] = "unsupported_geometry"

    return result

In [35]:
visual_df["parsed_s_region"] = (
    visual_df["s_region"]
    .apply(parse_s_region)
)


visual_df["region_geometry_type"] = (
    visual_df["parsed_s_region"]
    .apply(
        lambda parsed:
            parsed["geometry_type"]
    )
)


visual_df["region_parse_status"] = (
    visual_df["parsed_s_region"]
    .apply(
        lambda parsed:
            parsed["parse_status"]
    )
)

In [36]:
region_parse_summary = (
    visual_df[
        [
            "region_geometry_type",
            "region_parse_status",
        ]
    ]
    .value_counts(
        dropna=False,
    )
    .rename("archive_rows")
    .reset_index()
)


region_parse_summary

,region_geometry_type,region_parse_status,archive_rows
0,Polygon,parsed,528
1,Circle,parsed,4


In [37]:
unparsed_regions = (
    visual_df.loc[
        visual_df["region_parse_status"]
        != "parsed",
        [
            "region_geometry_type",
            "region_parse_status",
            "s_region",
        ],
    ]
    .drop_duplicates()
)


print(
    "Unparsed unique regions:",
    len(unparsed_regions),
)


unparsed_regions.head()

Unparsed unique regions: 0


,region_geometry_type,region_parse_status,s_region


In [38]:
def region_boundary_skycoord(parsed_region):
    """Return a SkyCoord boundary for a parsed spatial region."""

    geometry_type = parsed_region[
        "geometry_type"
    ]

    if (
        parsed_region["parse_status"]
        != "parsed"
    ):
        return None

    if geometry_type == "Polygon":
        vertices = np.asarray(
            parsed_region["vertices_deg"],
            dtype=float,
        )

        closed_vertices = np.vstack(
            [
                vertices,
                vertices[0],
            ]
        )

        return SkyCoord(
            ra=closed_vertices[:, 0] * u.deg,
            dec=closed_vertices[:, 1] * u.deg,
            frame="icrs",
        )

    if geometry_type == "Circle":
        centre = SkyCoord(
            ra=(
                parsed_region["centre_ra_deg"]
                * u.deg
            ),
            dec=(
                parsed_region["centre_dec_deg"]
                * u.deg
            ),
            frame="icrs",
        )

        position_angles = np.linspace(
            0,
            360,
            181,
        ) * u.deg

        return centre.directional_offset_by(
            position_angles,
            parsed_region["radius_deg"] * u.deg,
        )

    return None

In [39]:
def build_member_reference_coordinate(member_df):
    """Construct a local ICRS reference coordinate."""

    centre_df = (
        member_df[
            [
                "s_ra",
                "s_dec",
            ]
        ]
        .dropna()
        .drop_duplicates()
    )

    if centre_df.empty:
        raise ValueError(
            "No valid s_ra/s_dec coordinates "
            "were found for this Member OUS."
        )

    ra_radians = np.deg2rad(
        centre_df["s_ra"].astype(float)
    )

    mean_ra_radians = np.arctan2(
        np.mean(np.sin(ra_radians)),
        np.mean(np.cos(ra_radians)),
    )

    reference_ra_deg = (
        np.rad2deg(mean_ra_radians)
        % 360
    )

    reference_dec_deg = float(
        centre_df["s_dec"].astype(float).median()
    )

    return SkyCoord(
        ra=reference_ra_deg * u.deg,
        dec=reference_dec_deg * u.deg,
        frame="icrs",
    )

In [40]:
def skycoord_to_local_offsets(
    sky_coordinates,
    reference_coordinate,
):
    """Convert ICRS coordinates to local arcsecond offsets."""

    offset_frame = SkyOffsetFrame(
        origin=reference_coordinate
    )

    offset_coordinates = (
        sky_coordinates
        .transform_to(offset_frame)
    )

    x_offset_arcsec = (
        offset_coordinates.lon
        .wrap_at(180 * u.deg)
        .to_value(u.arcsec)
    )

    y_offset_arcsec = (
        offset_coordinates.lat
        .to_value(u.arcsec)
    )

    return (
        np.asarray(x_offset_arcsec),
        np.asarray(y_offset_arcsec),
    )

In [41]:
def prepare_member_spatial_records(member_uid):
    """Create unique source/footprint records for one Member OUS."""

    member_df = visual_df.loc[
        visual_df["member_ous_uid"]
        == member_uid
    ].copy()

    source_summary = (
        member_df
        .groupby(
            "obs_id_source",
            dropna=False,
        )
        .agg(
            spw_count=(
                "spw_identifier",
                "nunique",
            ),
            asdm_count=(
                "asdm_uid",
                "nunique",
            ),
            archive_row_count=(
                "obs_id",
                "size",
            ),
        )
        .reset_index()
    )

    spatial_records = (
        member_df[
            [
                "member_ous_uid",
                "obs_id_source",
                "target_name",
                "s_ra",
                "s_dec",
                "s_region",
                "parsed_s_region",
                "region_geometry_type",
                "region_parse_status",
                "is_mosaic",
            ]
        ]
        .drop_duplicates(
            subset=[
                "member_ous_uid",
                "obs_id_source",
                "s_ra",
                "s_dec",
                "s_region",
                "is_mosaic",
            ]
        )
        .merge(
            source_summary,
            on="obs_id_source",
            how="left",
        )
    )

    spatial_records["mosaic_state"] = (
        spatial_records["is_mosaic"]
        .astype(str)
        .map(
            {
                "T": "Mosaic",
                "F": "Non-mosaic",
            }
        )
        .fillna("Unknown")
    )

    return (
        member_df,
        spatial_records,
    )

In [42]:
test_member_df, test_spatial_records = (
    prepare_member_spatial_records(
        "uid://A001/X3833/X4d0"
    )
)


print(
    "Archive rows:",
    len(test_member_df),
)

print(
    "Unique spatial records:",
    len(test_spatial_records),
)


test_spatial_records[
    [
        "obs_id_source",
        "s_ra",
        "s_dec",
        "region_geometry_type",
        "mosaic_state",
        "spw_count",
        "asdm_count",
    ]
]

Archive rows: 112
Unique spatial records: 8


,obs_id_source,s_ra,s_dec,region_geometry_type,mosaic_state,spw_count,asdm_count
0,SDC24.433,279.169134,-7.666242,Polygon,Mosaic,14,1
1,SDC25.166,279.555374,-7.048491,Polygon,Mosaic,14,1
2,SDC28.333,280.712740,-4.038363,Polygon,Mosaic,14,1
3,SDC23.367,278.718915,-8.635148,Polygon,Mosaic,14,1
4,SDC24.489,279.608769,-7.825664,Polygon,Mosaic,14,1
5,SDC22.373,277.604829,-9.194196,Polygon,Mosaic,14,1
6,SDC24.633,278.907191,-7.308112,Polygon,Mosaic,14,1
7,SDC26.507,279.278409,-5.402636,Polygon,Mosaic,14,1


In [43]:
spatial_style_map = {
    "Non-mosaic": {
        "line": "#4C78A8",
        "fill": "rgba(76, 120, 168, 0.12)",
        "symbol": "circle",
    },
    "Mosaic": {
        "line": "#E45756",
        "fill": "rgba(228, 87, 86, 0.12)",
        "symbol": "diamond",
    },
    "Unknown": {
        "line": "#7F7F7F",
        "fill": "rgba(127, 127, 127, 0.10)",
        "symbol": "square",
    },
}

In [44]:
def build_spatial_footprint_figure(
    member_uid,
    show_footprints=True,
    show_centres=True,
    show_labels=False,
):
    """Build an interactive local sky-footprint plot."""

    member_df, spatial_records = (
        prepare_member_spatial_records(
            member_uid
        )
    )

    reference_coordinate = (
        build_member_reference_coordinate(
            member_df
        )
    )

    figure = go.Figure()

    all_x = []
    all_y = []

    footprint_legend_states = set()

    if show_footprints:
        for _, record in spatial_records.iterrows():
            boundary_skycoord = (
                region_boundary_skycoord(
                    record["parsed_s_region"]
                )
            )

            if boundary_skycoord is None:
                continue

            boundary_x, boundary_y = (
                skycoord_to_local_offsets(
                    boundary_skycoord,
                    reference_coordinate,
                )
            )

            all_x.extend(boundary_x.tolist())
            all_y.extend(boundary_y.tolist())

            mosaic_state = record[
                "mosaic_state"
            ]

            style = spatial_style_map.get(
                mosaic_state,
                spatial_style_map["Unknown"],
            )

            show_legend = (
                mosaic_state
                not in footprint_legend_states
            )

            footprint_legend_states.add(
                mosaic_state
            )

            hover_text = (
                f"<b>{record['obs_id_source']}</b><br>"
                f"Target: {record['target_name']}<br>"
                f"Geometry: "
                f"{record['region_geometry_type']}<br>"
                f"Mosaic state: {mosaic_state}<br>"
                f"SPWs: {record['spw_count']}<br>"
                f"ASDMs: {record['asdm_count']}<br>"
                f"RA: {float(record['s_ra']):.8f} deg<br>"
                f"Dec: {float(record['s_dec']):.8f} deg"
            )

            figure.add_trace(
                go.Scatter(
                    x=boundary_x,
                    y=boundary_y,
                    mode="lines",
                    fill="toself",
                    line={
                        "color": style["line"],
                        "width": 1.4,
                    },
                    fillcolor=style["fill"],
                    name=(
                        f"{mosaic_state} footprint"
                    ),
                    legendgroup=(
                        f"{mosaic_state}-footprint"
                    ),
                    showlegend=show_legend,
                    hoveron="fills+points",
                    hovertext=hover_text,
                    hoverinfo="text",
                )
            )

    if show_centres:
        centre_records = (
            spatial_records
            .dropna(
                subset=[
                    "s_ra",
                    "s_dec",
                ]
            )
            .drop_duplicates(
                subset=[
                    "obs_id_source",
                    "s_ra",
                    "s_dec",
                    "mosaic_state",
                ]
            )
        )

        for mosaic_state, state_df in (
            centre_records.groupby(
                "mosaic_state",
                dropna=False,
            )
        ):
            centre_skycoord = SkyCoord(
                ra=(
                    state_df["s_ra"]
                    .astype(float)
                    .to_numpy()
                    * u.deg
                ),
                dec=(
                    state_df["s_dec"]
                    .astype(float)
                    .to_numpy()
                    * u.deg
                ),
                frame="icrs",
            )

            centre_x, centre_y = (
                skycoord_to_local_offsets(
                    centre_skycoord,
                    reference_coordinate,
                )
            )

            all_x.extend(centre_x.tolist())
            all_y.extend(centre_y.tolist())

            style = spatial_style_map.get(
                mosaic_state,
                spatial_style_map["Unknown"],
            )

            custom_data = state_df[
                [
                    "target_name",
                    "s_ra",
                    "s_dec",
                    "region_geometry_type",
                    "spw_count",
                    "asdm_count",
                ]
            ].to_numpy()

            marker_mode = (
                "markers+text"
                if show_labels
                else "markers"
            )

            figure.add_trace(
                go.Scatter(
                    x=centre_x,
                    y=centre_y,
                    mode=marker_mode,
                    text=state_df[
                        "obs_id_source"
                    ],
                    textposition="top center",
                    marker={
                        "size": 10,
                        "symbol": style["symbol"],
                        "color": style["line"],
                        "line": {
                            "width": 1,
                            "color": "gray",
                        },
                    },
                    name=(
                        f"{mosaic_state} centre"
                    ),
                    legendgroup=(
                        f"{mosaic_state}-centre"
                    ),
                    customdata=custom_data,
                    hovertemplate=(
                        "<b>%{text}</b><br>"
                        "Target: %{customdata[0]}<br>"
                        "RA: %{customdata[1]:.8f} deg<br>"
                        "Dec: %{customdata[2]:.8f} deg<br>"
                        "Geometry: %{customdata[3]}<br>"
                        "SPWs: %{customdata[4]}<br>"
                        "ASDMs: %{customdata[5]}"
                        "<extra></extra>"
                    ),
                )
            )

    if all_x and all_y:
        minimum_x = min(all_x)
        maximum_x = max(all_x)
        minimum_y = min(all_y)
        maximum_y = max(all_y)

        largest_span = max(
            maximum_x - minimum_x,
            maximum_y - minimum_y,
            1.0,
        )

        padding = largest_span * 0.08

        x_axis_range = [
            maximum_x + padding,
            minimum_x - padding,
        ]

        y_axis_range = [
            minimum_y - padding,
            maximum_y + padding,
        ]

    else:
        x_axis_range = [1, -1]
        y_axis_range = [-1, 1]

    figure.update_layout(
        title=(
            "Archive sky centres and spatial footprints — "
            + uid_to_case.get(
                member_uid,
                member_uid,
            )
        ),
        width=1050,
        height=720,
        xaxis={
            "title": (
                "ΔRA cos(Dec) "
                "(arcsec; east is left)"
            ),
            "range": x_axis_range,
            "zeroline": True,
            "showgrid": True,
            "constrain": "domain",
        },
        yaxis={
            "title": (
                "ΔDec (arcsec; north is up)"
            ),
            "range": y_axis_range,
            "zeroline": True,
            "showgrid": True,
            "scaleanchor": "x",
            "scaleratio": 1,
            "constrain": "domain",
        },
        legend={
            "title": {
                "text": "Archive spatial metadata"
            },
        },
        hovermode="closest",
        uirevision=member_uid,
        margin={
            "l": 90,
            "r": 40,
            "t": 90,
            "b": 80,
        },
    )

    return (
        figure,
        reference_coordinate,
        spatial_records,
    )

In [46]:
spatial_member_dropdown = ipywidgets.Dropdown(
    options=[
        (case_name, member_uid)
        for case_name, member_uid
        in representative_members.items()
    ],
    value=next(
        iter(representative_members.values())
    ),
    description="Member:",
    layout=ipywidgets.Layout(
        width="720px",
    ),
    style={
        "description_width": "80px",
    },
)


show_footprints_checkbox = ipywidgets.Checkbox(
    value=True,
    description="Show footprints",
    indent=False,
)


show_centres_checkbox = ipywidgets.Checkbox(
    value=True,
    description="Show centres",
    indent=False,
)


show_labels_checkbox = ipywidgets.Checkbox(
    value=False,
    description="Show source labels",
    indent=False,
)


spatial_output = ipywidgets.Output()

In [47]:
def render_spatial_view(change=None):
    """Render selected Member OUS spatial metadata."""

    member_uid = spatial_member_dropdown.value

    (
        spatial_figure,
        reference_coordinate,
        spatial_records,
    ) = build_spatial_footprint_figure(
        member_uid=member_uid,
        show_footprints=(
            show_footprints_checkbox.value
        ),
        show_centres=(
            show_centres_checkbox.value
        ),
        show_labels=(
            show_labels_checkbox.value
        ),
    )

    with spatial_output:
        clear_output(wait=True)

        print(
            uid_to_case.get(
                member_uid,
                member_uid,
            )
        )

        print(
            "Reference coordinate: "
            f"RA = {reference_coordinate.ra.deg:.8f} deg, "
            f"Dec = {reference_coordinate.dec.deg:.8f} deg"
        )

        print(
            "Source contexts:",
            spatial_records[
                "obs_id_source"
            ].nunique(),
            "| Centres:",
            spatial_records[
                ["s_ra", "s_dec"]
            ].drop_duplicates().shape[0],
            "| Footprints:",
            spatial_records[
                "s_region"
            ].nunique(),
        )

        print(
            "Mosaic states:",
            sorted(
                spatial_records[
                    "mosaic_state"
                ].unique().tolist()
            ),
            "| Geometries:",
            sorted(
                spatial_records[
                    "region_geometry_type"
                ].unique().tolist()
            ),
        )

        display(spatial_figure)

In [62]:
spatial_member_dropdown.observe(
    render_spatial_view,
    names="value",
)


show_footprints_checkbox.observe(
    render_spatial_view,
    names="value",
)


show_centres_checkbox.observe(
    render_spatial_view,
    names="value",
)


show_labels_checkbox.observe(
    render_spatial_view,
    names="value",
)

In [63]:
spatial_controls = ipywidgets.VBox(
    [
        spatial_member_dropdown,
        ipywidgets.HBox(
            [
                show_footprints_checkbox,
                show_centres_checkbox,
                show_labels_checkbox,
            ]
        ),
    ]
)


display(
    spatial_controls,
    spatial_output,
)


render_spatial_view()

Output()

## 5. All-Sky Context and Local Footprint Detail

The all-sky view locates each representative Member OUS on the celestial
sphere. The selected point represents the Member-level reference direction,
not the angular size of its spatial footprint.

Two global context views are provided:

- a Mollweide projection following common astronomical all-sky conventions;
- an optional rotatable three-dimensional celestial sphere.

The detailed local panel remains necessary because ALMA footprints are usually
far too small to be visible at an all-sky scale.

In [50]:
def classify_member_mosaic_state(member_df):
    """Derive a display-only Member mosaic category."""

    states = set(
        member_df["is_mosaic"]
        .dropna()
        .astype(str)
        .tolist()
    )

    if states == {"T"}:
        return "Mosaic"

    if states == {"F"}:
        return "Non-mosaic"

    if states == {"T", "F"}:
        return "Mixed"

    return "Unknown"

In [51]:
all_sky_rows = []


for case_name, member_uid in (
    representative_members.items()
):
    member_df = visual_df.loc[
        visual_df["member_ous_uid"]
        == member_uid
    ].copy()

    reference_coordinate = (
        build_member_reference_coordinate(
            member_df
        )
    )

    reference_ra_deg = float(
        reference_coordinate.ra.deg
    )

    reference_dec_deg = float(
        reference_coordinate.dec.deg
    )

    # Plotly maps longitude increasing to the right.
    # Negating RA makes astronomical east appear on the left.
    plot_longitude_deg = (
        (180 - reference_ra_deg) % 360
    ) - 180

    target_names = sorted(
        member_df["target_name"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    target_display = ", ".join(
        target_names[:3]
    )

    if len(target_names) > 3:
        target_display += (
            f" (+{len(target_names) - 3} more)"
        )

    all_sky_rows.append(
        {
            "case": case_name,
            "member_ous_uid": member_uid,
            "reference_ra_deg":
                reference_ra_deg,
            "reference_dec_deg":
                reference_dec_deg,
            "plot_longitude_deg":
                plot_longitude_deg,
            "ra_hms": (
                reference_coordinate.ra
                .to_string(
                    unit=u.hourangle,
                    sep=":",
                    precision=1,
                    pad=True,
                )
            ),
            "dec_dms": (
                reference_coordinate.dec
                .to_string(
                    unit=u.deg,
                    sep=":",
                    precision=1,
                    alwayssign=True,
                    pad=True,
                )
            ),
            "source_count": (
                member_df[
                    "obs_id_source"
                ].nunique()
            ),
            "spw_count": (
                member_df[
                    "spw_identifier"
                ].nunique()
            ),
            "asdm_count": (
                member_df[
                    "asdm_uid"
                ].nunique()
            ),
            "archive_rows": len(member_df),
            "mosaic_class":
                classify_member_mosaic_state(
                    member_df
                ),
            "targets": target_display,
        }
    )


all_sky_member_df = pd.DataFrame(
    all_sky_rows
)


all_sky_member_df

,case,member_ous_uid,reference_ra_deg,reference_dec_deg,plot_longitude_deg,ra_hms,dec_dms,source_count,spw_count,asdm_count,archive_rows,mosaic_class,targets
0,Simple — 1 source × 4 SPWs,uid://A001/X15a1/Xe3b,79.001988,-9.809838,-79.001988,05:16:00.5,-09:48:35.4,1,4,1,4,Non-mosaic,HD34282
1,Multi-ASDM — 2 sources × 4 SPWs,uid://A001/X3833/X1022,357.428091,-56.631489,2.571909,23:49:42.7,-56:37:53.4,2,4,2,8,Mosaic,"SPT2349-56_Core, SPT2349-56_N1-N2"
2,Large source grid — 36 sources × 4 SPWs,uid://A001/X3819/X186,274.929142,-18.245556,85.070858,18:19:43.0,-18:14:44.0,36,4,1,144,Non-mosaic,"008.672-00.682, 008.672-00.682_OFF_0, 008.872-00.492 (+33 more)"
3,Large spectral grid — 16 sources × 16 SPWs,uid://A001/X3788/Xc584,83.897251,-5.099144,-83.897251,05:35:35.3,-05:05:56.9,16,16,1,256,Non-mosaic,"HOPS-108, HOPS-153, HOPS-370 (+13 more)"
4,Mosaic — 8 sources × 14 SPWs,uid://A001/X3833/X4d0,279.194423,-7.487177,80.805577,18:36:46.7,-07:29:13.8,8,14,1,112,Mosaic,"SDC22.373, SDC23.367, SDC24.433 (+5 more)"
5,Mixed mosaic-state case,uid://A001/X383d/X43a,13.686975,-37.651463,-13.686975,00:54:44.9,-37:39:05.3,2,4,1,8,Mixed,"NGC_0300, NGC_0300_FluxRef"


In [52]:
all_sky_member_df[
    [
        "case",
        "reference_ra_deg",
        "reference_dec_deg",
        "plot_longitude_deg",
        "ra_hms",
        "dec_dms",
        "mosaic_class",
    ]
]

,case,reference_ra_deg,reference_dec_deg,plot_longitude_deg,ra_hms,dec_dms,mosaic_class
0,Simple — 1 source × 4 SPWs,79.001988,-9.809838,-79.001988,05:16:00.5,-09:48:35.4,Non-mosaic
1,Multi-ASDM — 2 sources × 4 SPWs,357.428091,-56.631489,2.571909,23:49:42.7,-56:37:53.4,Mosaic
2,Large source grid — 36 sources × 4 SPWs,274.929142,-18.245556,85.070858,18:19:43.0,-18:14:44.0,Non-mosaic
3,Large spectral grid — 16 sources × 16 SPWs,83.897251,-5.099144,-83.897251,05:35:35.3,-05:05:56.9,Non-mosaic
4,Mosaic — 8 sources × 14 SPWs,279.194423,-7.487177,80.805577,18:36:46.7,-07:29:13.8,Mosaic
5,Mixed mosaic-state case,13.686975,-37.651463,-13.686975,00:54:44.9,-37:39:05.3,Mixed


In [53]:
all_sky_colour_map = {
    "Non-mosaic": "#4C78A8",
    "Mosaic": "#E45756",
    "Mixed": "#B279A2",
    "Unknown": "#7F7F7F",
}

In [54]:
def build_mollweide_figure(
    selected_member_uid,
):
    """Build an astronomical all-sky context view."""

    figure = go.Figure()

    for mosaic_class, state_df in (
        all_sky_member_df.groupby(
            "mosaic_class",
            dropna=False,
        )
    ):
        custom_data = state_df[
            [
                "case",
                "member_ous_uid",
                "ra_hms",
                "dec_dms",
                "reference_ra_deg",
                "reference_dec_deg",
                "source_count",
                "spw_count",
                "asdm_count",
                "archive_rows",
                "targets",
            ]
        ].to_numpy()

        figure.add_trace(
            go.Scattergeo(
                lon=state_df[
                    "plot_longitude_deg"
                ],
                lat=state_df[
                    "reference_dec_deg"
                ],
                mode="markers",
                name=mosaic_class,
                marker={
                    "size": 11,
                    "color": (
                        all_sky_colour_map.get(
                            mosaic_class,
                            "#7F7F7F",
                        )
                    ),
                    "line": {
                        "width": 0.8,
                        "color": "gray",
                    },
                },
                customdata=custom_data,
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Member: %{customdata[1]}<br>"
                    "RA: %{customdata[2]} "
                    "(%{customdata[4]:.6f}°)<br>"
                    "Dec: %{customdata[3]} "
                    "(%{customdata[5]:.6f}°)<br>"
                    "Sources: %{customdata[6]}<br>"
                    "SPWs: %{customdata[7]}<br>"
                    "ASDMs: %{customdata[8]}<br>"
                    "Archive rows: %{customdata[9]}<br>"
                    "Targets: %{customdata[10]}"
                    "<extra></extra>"
                ),
            )
        )

    selected_df = all_sky_member_df.loc[
        all_sky_member_df[
            "member_ous_uid"
        ] == selected_member_uid
    ]

    figure.add_trace(
        go.Scattergeo(
            lon=selected_df[
                "plot_longitude_deg"
            ],
            lat=selected_df[
                "reference_dec_deg"
            ],
            mode="markers",
            name="Selected Member",
            marker={
                "size": 19,
                "symbol": "star",
                "color": "#F2CF5B",
                "line": {
                    "width": 2,
                    "color": "#333333",
                },
            },
            hoverinfo="skip",
        )
    )

    figure.update_layout(
        title=(
            "Representative Member OUS positions "
            "on the celestial sphere"
        ),
        width=1050,
        height=590,
        geo={
            "projection": {
                "type": "mollweide",
                "rotation": {
                    "lon": 0,
                },
            },
            "showframe": True,
            "showcoastlines": False,
            "showland": False,
            "showocean": False,
            "showlakes": False,
            "bgcolor": "rgba(0,0,0,0)",
            "lonaxis": {
                "showgrid": True,
                "gridwidth": 0.6,
                "gridcolor": "rgba(120,120,120,0.45)",
                "dtick": 30,
            },
            "lataxis": {
                "showgrid": True,
                "gridwidth": 0.6,
                "gridcolor": "rgba(120,120,120,0.45)",
                "dtick": 30,
            },
        },
        legend={
            "title": {
                "text": "Member mosaic category"
            },
        },
        margin={
            "l": 30,
            "r": 30,
            "t": 80,
            "b": 55,
        },
        annotations=[
            {
                "text": (
                    "Right Ascension increases "
                    "towards the left"
                ),
                "x": 0.5,
                "y": -0.04,
                "xref": "paper",
                "yref": "paper",
                "showarrow": False,
            }
        ],
    )

    return figure

In [56]:
def radec_to_unit_sphere(
    ra_deg,
    dec_deg,
    radius=1.03,
):
    """Convert RA/Dec to Cartesian celestial-sphere coordinates."""

    ra_radians = np.deg2rad(
        np.asarray(ra_deg, dtype=float)
    )

    dec_radians = np.deg2rad(
        np.asarray(dec_deg, dtype=float)
    )

    x = (
        radius
        * np.cos(dec_radians)
        * np.cos(ra_radians)
    )

    y = (
        radius
        * np.cos(dec_radians)
        * np.sin(ra_radians)
    )

    z = (
        radius
        * np.sin(dec_radians)
    )

    return x, y, z

In [57]:
def build_celestial_sphere_figure(
    selected_member_uid,
):
    """Build a rotatable 3D celestial sphere."""

    figure = go.Figure()

    longitude_grid = np.linspace(
        0,
        2 * np.pi,
        100,
    )

    latitude_grid = np.linspace(
        -np.pi / 2,
        np.pi / 2,
        60,
    )

    longitude_mesh, latitude_mesh = (
        np.meshgrid(
            longitude_grid,
            latitude_grid,
        )
    )

    sphere_x = (
        np.cos(latitude_mesh)
        * np.cos(longitude_mesh)
    )

    sphere_y = (
        np.cos(latitude_mesh)
        * np.sin(longitude_mesh)
    )

    sphere_z = np.sin(latitude_mesh)

    figure.add_trace(
        go.Surface(
            x=sphere_x,
            y=sphere_y,
            z=sphere_z,
            surfacecolor=np.zeros_like(
                sphere_x
            ),
            colorscale=[
                [0, "#A8C5E5"],
                [1, "#A8C5E5"],
            ],
            opacity=0.10,
            showscale=False,
            hoverinfo="skip",
            name="Celestial sphere",
        )
    )

    # Declination grid circles
    for dec_deg in [
        -60,
        -30,
        0,
        30,
        60,
    ]:
        ra_values = np.linspace(
            0,
            360,
            181,
        )

        dec_values = np.full_like(
            ra_values,
            dec_deg,
            dtype=float,
        )

        grid_x, grid_y, grid_z = (
            radec_to_unit_sphere(
                ra_values,
                dec_values,
                radius=1.005,
            )
        )

        figure.add_trace(
            go.Scatter3d(
                x=grid_x,
                y=grid_y,
                z=grid_z,
                mode="lines",
                line={
                    "width": 1,
                    "color": (
                        "rgba(110,110,110,0.35)"
                    ),
                },
                showlegend=False,
                hoverinfo="skip",
            )
        )

    # RA great-circle grid
    for ra_deg in range(
        0,
        360,
        30,
    ):
        dec_values = np.linspace(
            -90,
            90,
            121,
        )

        ra_values = np.full_like(
            dec_values,
            ra_deg,
            dtype=float,
        )

        grid_x, grid_y, grid_z = (
            radec_to_unit_sphere(
                ra_values,
                dec_values,
                radius=1.005,
            )
        )

        figure.add_trace(
            go.Scatter3d(
                x=grid_x,
                y=grid_y,
                z=grid_z,
                mode="lines",
                line={
                    "width": 1,
                    "color": (
                        "rgba(110,110,110,0.25)"
                    ),
                },
                showlegend=False,
                hoverinfo="skip",
            )
        )

    for mosaic_class, state_df in (
        all_sky_member_df.groupby(
            "mosaic_class",
            dropna=False,
        )
    ):
        point_x, point_y, point_z = (
            radec_to_unit_sphere(
                state_df[
                    "reference_ra_deg"
                ],
                state_df[
                    "reference_dec_deg"
                ],
            )
        )

        custom_data = state_df[
            [
                "case",
                "member_ous_uid",
                "ra_hms",
                "dec_dms",
                "source_count",
                "spw_count",
                "asdm_count",
                "targets",
            ]
        ].to_numpy()

        figure.add_trace(
            go.Scatter3d(
                x=point_x,
                y=point_y,
                z=point_z,
                mode="markers",
                name=mosaic_class,
                marker={
                    "size": 7,
                    "color": (
                        all_sky_colour_map.get(
                            mosaic_class,
                            "#7F7F7F",
                        )
                    ),
                    "line": {
                        "width": 0.8,
                        "color": "#333333",
                    },
                },
                customdata=custom_data,
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Member: %{customdata[1]}<br>"
                    "RA: %{customdata[2]}<br>"
                    "Dec: %{customdata[3]}<br>"
                    "Sources: %{customdata[4]}<br>"
                    "SPWs: %{customdata[5]}<br>"
                    "ASDMs: %{customdata[6]}<br>"
                    "Targets: %{customdata[7]}"
                    "<extra></extra>"
                ),
            )
        )

    selected_df = all_sky_member_df.loc[
        all_sky_member_df[
            "member_ous_uid"
        ] == selected_member_uid
    ]

    selected_x, selected_y, selected_z = (
        radec_to_unit_sphere(
            selected_df["reference_ra_deg"],
            selected_df["reference_dec_deg"],
            radius=1.05,
        )
    )

    figure.add_trace(
        go.Scatter3d(
            x=selected_x,
            y=selected_y,
            z=selected_z,
            mode="markers",
            name="Selected Member",
            marker={
                "size": 11,
                "symbol": "diamond",
                "color": "#F2CF5B",
                "line": {
                    "width": 2,
                    "color": "#333333",
                },
            },
            hoverinfo="skip",
        )
    )

    figure.update_layout(
        title=(
            "Rotatable celestial sphere — "
            "representative Member OUS positions"
        ),
        width=950,
        height=720,
        scene={
            "aspectmode": "data",
            "xaxis": {
                "visible": False,
            },
            "yaxis": {
                "visible": False,
            },
            "zaxis": {
                "visible": False,
            },
            "camera": {
                "eye": {
                    "x": 1.45,
                    "y": 1.45,
                    "z": 0.85,
                }
            },
            "bgcolor": "rgba(0,0,0,0)",
        },
        legend={
            "title": {
                "text": "Member mosaic category"
            },
        },
        margin={
            "l": 20,
            "r": 20,
            "t": 80,
            "b": 20,
        },
    )

    return figure

In [58]:
combined_member_dropdown = (
    ipywidgets.Dropdown(
        options=[
            (case_name, member_uid)
            for case_name, member_uid
            in representative_members.items()
        ],
        value=next(
            iter(
                representative_members.values()
            )
        ),
        description="Member:",
        layout=ipywidgets.Layout(
            width="720px",
        ),
        style={
            "description_width": "80px",
        },
    )
)


sky_context_toggle = (
    ipywidgets.ToggleButtons(
        options=[
            (
                "Mollweide all-sky",
                "mollweide",
            ),
            (
                "3D celestial sphere",
                "sphere",
            ),
        ],
        value="mollweide",
        description="Context:",
        style={
            "description_width": "80px",
        },
    )
)


combined_show_footprints = (
    ipywidgets.Checkbox(
        value=True,
        description="Show footprints",
        indent=False,
    )
)


combined_show_centres = (
    ipywidgets.Checkbox(
        value=True,
        description="Show centres",
        indent=False,
    )
)


combined_show_labels = (
    ipywidgets.Checkbox(
        value=False,
        description="Show source labels",
        indent=False,
    )
)


combined_sky_output = ipywidgets.Output()

In [59]:
def render_combined_sky_view(change=None):
    """Render global context and local footprint detail."""

    member_uid = (
        combined_member_dropdown.value
    )

    context_type = (
        sky_context_toggle.value
    )

    if context_type == "mollweide":
        context_figure = (
            build_mollweide_figure(
                member_uid
            )
        )
    else:
        context_figure = (
            build_celestial_sphere_figure(
                member_uid
            )
        )

    (
        local_figure,
        reference_coordinate,
        spatial_records,
    ) = build_spatial_footprint_figure(
        member_uid=member_uid,
        show_footprints=(
            combined_show_footprints.value
        ),
        show_centres=(
            combined_show_centres.value
        ),
        show_labels=(
            combined_show_labels.value
        ),
    )

    with combined_sky_output:
        clear_output(wait=True)

        selected_row = (
            all_sky_member_df.loc[
                all_sky_member_df[
                    "member_ous_uid"
                ] == member_uid
            ]
            .iloc[0]
        )

        print(selected_row["case"])

        print(
            "Reference direction: "
            f"RA {selected_row['ra_hms']}, "
            f"Dec {selected_row['dec_dms']}"
        )

        print(
            f"{selected_row['source_count']} sources | "
            f"{selected_row['spw_count']} SPWs | "
            f"{selected_row['asdm_count']} ASDMs | "
            f"{selected_row['archive_rows']} Archive rows"
        )

        display(context_figure)
        display(local_figure)

In [60]:
for widget in [
    combined_member_dropdown,
    sky_context_toggle,
    combined_show_footprints,
    combined_show_centres,
    combined_show_labels,
]:
    widget.observe(
        render_combined_sky_view,
        names="value",
    )

In [61]:
combined_controls = ipywidgets.VBox(
    [
        combined_member_dropdown,
        sky_context_toggle,
        ipywidgets.HBox(
            [
                combined_show_footprints,
                combined_show_centres,
                combined_show_labels,
            ]
        ),
    ]
)


display(
    combined_controls,
    combined_sky_output,
)


render_combined_sky_view()

Output()

## 6. Field Stability and Candidate Data Ownership

This visualization evaluates the exact-value stability of Archive fields under
four candidate grouping levels:

1. Member OUS
2. Source Context
3. Logical SPW
4. Source–SPW Record

For each field and grouping level, the stability score is the fraction of
available groups that contain no more than one distinct value.

Missing-value availability and completeness are calculated separately so that
an entirely missing field is not incorrectly interpreted as stable.

This is an empirical assessment of the selected Archive records. It is not an
official ALMA schema definition, and floating-point differences are initially
evaluated exactly without a scientific tolerance.

In [64]:
requested_stability_fields = [
    "proposal_id",
    "group_ous_uid",
    "asdm_uid",
    "obs_id",
    "target_name",
    "s_ra",
    "s_dec",
    "s_region",
    "is_mosaic",
    "frequency",
    "bandwidth",
    "frequency_support",
    "spatial_resolution",
    "sensitivity_10kms",
    "cont_sensitivity_bandwidth",
    "antenna_arrays",
    "t_min",
    "t_max",
]


available_stability_fields = [
    field
    for field in requested_stability_fields
    if field in visual_df.columns
]


missing_stability_fields = [
    field
    for field in requested_stability_fields
    if field not in visual_df.columns
]


print(
    "Fields included:",
    len(available_stability_fields),
)

print(
    "Fields unavailable:",
    missing_stability_fields,
)

Fields included: 18
Fields unavailable: []


In [66]:
stability_grouping_levels = {
    "Member OUS": [
        "member_ous_uid",
    ],

    "Source Context": [
        "member_ous_uid",
        "obs_id_source",
    ],

    "Logical SPW": [
        "member_ous_uid",
        "spw_identifier",
    ],

    "Source × SPW Record": [
        "member_ous_uid",
        "obs_id_source",
        "spw_identifier",
    ],
}

In [67]:
stability_analysis_df = (
    visual_df.copy()
)


def normalize_blank_value(value):
    """Convert blank strings to Pandas missing values."""

    if isinstance(value, str):
        if not value.strip():
            return pd.NA

    return value


for field in available_stability_fields:
    stability_analysis_df[field] = (
        stability_analysis_df[field]
        .map(normalize_blank_value)
    )

In [68]:
print(
    "Blank group_ous_uid values:",
    (
        stability_analysis_df[
            "group_ous_uid"
        ]
        .astype("string")
        .str.strip()
        .eq("")
        .sum()
    ),
)


print(
    "Missing group_ous_uid values:",
    stability_analysis_df[
        "group_ous_uid"
    ].isna().sum(),
)

Blank group_ous_uid values: 0
Missing group_ous_uid values: 0


In [69]:
def evaluate_field_stability(
    dataframe,
    field,
    level_name,
    grouping_columns,
):
    """Evaluate one field under one candidate grouping level."""

    grouped_field = (
        dataframe
        .groupby(
            grouping_columns,
            dropna=False,
            sort=False,
        )[field]
    )

    group_sizes = grouped_field.size()

    nonmissing_counts = (
        grouped_field.count()
    )

    unique_counts = (
        grouped_field.nunique(
            dropna=False
        )
    )

    available_groups = (
        nonmissing_counts > 0
    )

    complete_groups = (
        nonmissing_counts == group_sizes
    )

    if available_groups.any():
        available_unique_counts = (
            unique_counts.loc[
                available_groups
            ]
        )

        stable_groups = (
            available_unique_counts <= 1
        )

        stability_fraction = float(
            stable_groups.mean()
        )

        maximum_unique_values = int(
            available_unique_counts.max()
        )

    else:
        stability_fraction = np.nan
        maximum_unique_values = np.nan

    availability_fraction = float(
        available_groups.mean()
    )

    completeness_fraction = float(
        complete_groups.mean()
    )

    return {
        "field": field,
        "level": level_name,
        "stability_fraction":
            stability_fraction,
        "availability_fraction":
            availability_fraction,
        "completeness_fraction":
            completeness_fraction,
        "group_count":
            int(len(group_sizes)),
        "available_group_count":
            int(available_groups.sum()),
        "maximum_unique_values":
            maximum_unique_values,
    }

In [70]:
def calculate_stability_table(
    dataframe,
):
    """Calculate stability for all fields and grouping levels."""

    stability_rows = []

    for field in available_stability_fields:
        for (
            level_name,
            grouping_columns,
        ) in stability_grouping_levels.items():

            result = (
                evaluate_field_stability(
                    dataframe=dataframe,
                    field=field,
                    level_name=level_name,
                    grouping_columns=(
                        grouping_columns
                    ),
                )
            )

            stability_rows.append(result)

    return pd.DataFrame(
        stability_rows
    )

In [71]:
all_member_stability_df = (
    calculate_stability_table(
        stability_analysis_df
    )
)


all_member_stability_df.head(12)

,field,level,stability_fraction,availability_fraction,completeness_fraction,group_count,available_group_count,maximum_unique_values
0,proposal_id,Member OUS,1.000000,1.0,1.0,6,6,1
1,proposal_id,Source Context,1.000000,1.0,1.0,65,65,1
2,proposal_id,Logical SPW,1.000000,1.0,1.0,46,46,1
3,proposal_id,Source × SPW Record,1.000000,1.0,1.0,532,532,1
4,group_ous_uid,Member OUS,1.000000,1.0,1.0,6,6,1
5,group_ous_uid,Source Context,1.000000,1.0,1.0,65,65,1
6,group_ous_uid,Logical SPW,1.000000,1.0,1.0,46,46,1
7,group_ous_uid,Source × SPW Record,1.000000,1.0,1.0,532,532,1
8,asdm_uid,Member OUS,0.833333,1.0,1.0,6,6,2
9,asdm_uid,Source Context,1.000000,1.0,1.0,65,65,1


In [72]:
expected_stability_rows = (
    len(available_stability_fields)
    * len(stability_grouping_levels)
)


print(
    "Expected stability rows:",
    expected_stability_rows,
)

print(
    "Calculated stability rows:",
    len(all_member_stability_df),
)

print(
    "Complete stability table:",
    (
        len(all_member_stability_df)
        == expected_stability_rows
    ),
)

Expected stability rows: 72
Calculated stability rows: 72
Complete stability table: True


In [73]:
def build_stability_matrices(
    stability_table,
):
    """Pivot long-form stability results into heatmap matrices."""

    field_order = (
        available_stability_fields
    )

    level_order = list(
        stability_grouping_levels.keys()
    )

    stability_matrix = (
        stability_table
        .pivot(
            index="field",
            columns="level",
            values="stability_fraction",
        )
        .reindex(
            index=field_order,
            columns=level_order,
        )
    )

    availability_matrix = (
        stability_table
        .pivot(
            index="field",
            columns="level",
            values="availability_fraction",
        )
        .reindex(
            index=field_order,
            columns=level_order,
        )
    )

    completeness_matrix = (
        stability_table
        .pivot(
            index="field",
            columns="level",
            values="completeness_fraction",
        )
        .reindex(
            index=field_order,
            columns=level_order,
        )
    )

    group_count_matrix = (
        stability_table
        .pivot(
            index="field",
            columns="level",
            values="group_count",
        )
        .reindex(
            index=field_order,
            columns=level_order,
        )
    )

    maximum_unique_matrix = (
        stability_table
        .pivot(
            index="field",
            columns="level",
            values="maximum_unique_values",
        )
        .reindex(
            index=field_order,
            columns=level_order,
        )
    )

    return {
        "stability": stability_matrix,
        "availability": availability_matrix,
        "completeness": completeness_matrix,
        "group_count": group_count_matrix,
        "maximum_unique":
            maximum_unique_matrix,
    }

In [74]:
def build_field_stability_heatmap(
    stability_table,
    scope_label,
):
    """Build an interactive field-stability heatmap."""

    matrices = build_stability_matrices(
        stability_table
    )

    stability_matrix = matrices[
        "stability"
    ]

    availability_matrix = matrices[
        "availability"
    ]

    completeness_matrix = matrices[
        "completeness"
    ]

    group_count_matrix = matrices[
        "group_count"
    ]

    maximum_unique_matrix = matrices[
        "maximum_unique"
    ]

    text_matrix = (
        stability_matrix
        .map(
            lambda value:
                "N/A"
                if pd.isna(value)
                else f"{value:.0%}"
        )
    )

    custom_data = np.dstack(
        [
            availability_matrix.to_numpy(
                dtype=float
            ),
            completeness_matrix.to_numpy(
                dtype=float
            ),
            group_count_matrix.to_numpy(
                dtype=float
            ),
            maximum_unique_matrix.to_numpy(
                dtype=float
            ),
        ]
    )

    figure = go.Figure(
        data=go.Heatmap(
            z=stability_matrix.to_numpy(
                dtype=float
            ),
            x=stability_matrix.columns,
            y=stability_matrix.index,
            zmin=0,
            zmax=1,
            colorscale="RdYlGn",
            text=text_matrix.to_numpy(),
            texttemplate="%{text}",
            customdata=custom_data,
            xgap=2,
            ygap=2,
            colorbar={
                "title": {
                    "text": (
                        "Exact-value<br>stability"
                    )
                },
                "tickvals": [
                    0,
                    0.25,
                    0.5,
                    0.75,
                    1,
                ],
                "ticktext": [
                    "0%",
                    "25%",
                    "50%",
                    "75%",
                    "100%",
                ],
            },
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Candidate level: %{x}<br>"
                "Stability: %{z:.1%}<br>"
                "Availability: "
                "%{customdata[0]:.1%}<br>"
                "Completeness: "
                "%{customdata[1]:.1%}<br>"
                "Groups evaluated: "
                "%{customdata[2]:.0f}<br>"
                "Maximum unique values: "
                "%{customdata[3]:.0f"
                "}<extra></extra>"
            ),
        )
    )

    figure.update_layout(
        title=(
            "Exact field stability by "
            "candidate data-model level — "
            + scope_label
        ),
        width=1050,
        height=max(
            720,
            39 * len(
                stability_matrix.index
            ) + 170,
        ),
        xaxis={
            "title": (
                "Candidate grouping level"
            ),
            "side": "top",
        },
        yaxis={
            "title": "Archive field",
            "autorange": "reversed",
        },
        margin={
            "l": 210,
            "r": 100,
            "t": 140,
            "b": 70,
        },
    )

    return figure

In [75]:
hovertemplate=(
    "<b>%{y}</b><br>"
    "Candidate level: %{x}<br>"
    "Stability: %{z:.1%}<br>"
    "Availability: %{customdata[0]:.1%}<br>"
    "Completeness: %{customdata[1]:.1%}<br>"
    "Groups evaluated: %{customdata[2]:.0f}<br>"
    "Maximum unique values: %{customdata[3]:.0f}"
    "<extra></extra>"
),

In [ ]:
stability_scope_options = [
    (
        "All six representative Members",
        "__all__",
    ),
    *[
        (case_name, member_uid)
        for case_name, member_uid
        in representative_members.items()
    ],
]


stability_scope_dropdown = (
    ipywidgets.Dropdown(
        options=stability_scope_options,
        value="__all__",
        description="Scope:",
        layout=ipywidgets.Layout(
            width="720px",
        ),
        style={
            "description_width": "80px",
        },
    )
)


stability_output = ipywidgets.Output()

In [78]:
def render_field_stability_view(
    change=None,
):
    """Render stability heatmap for selected scope."""

    selected_scope = (
        stability_scope_dropdown.value
    )

    if selected_scope == "__all__":
        selected_df = (
            stability_analysis_df
        )

        scope_label = (
            "all six representative Members"
        )

    else:
        selected_df = (
            stability_analysis_df.loc[
                stability_analysis_df[
                    "member_ous_uid"
                ] == selected_scope
            ]
        )

        scope_label = uid_to_case.get(
            selected_scope,
            selected_scope,
        )

    selected_stability_table = (
        calculate_stability_table(
            selected_df
        )
    )

    selected_figure = (
        build_field_stability_heatmap(
            selected_stability_table,
            scope_label,
        )
    )

    with stability_output:
        clear_output(wait=True)

        print(scope_label)

        print(
            "Archive rows:",
            len(selected_df),
            "| Members:",
            selected_df[
                "member_ous_uid"
            ].nunique(),
            "| Source contexts:",
            selected_df[
                [
                    "member_ous_uid",
                    "obs_id_source",
                ]
            ].drop_duplicates().shape[0],
            "| Logical SPWs:",
            selected_df[
                [
                    "member_ous_uid",
                    "spw_identifier",
                ]
            ].drop_duplicates().shape[0],
        )

        display(selected_figure)

In [79]:
stability_scope_dropdown.observe(
    render_field_stability_view,
    names="value",
)


display(
    stability_scope_dropdown,
    stability_output,
)


render_field_stability_view()

Dropdown(description='Scope:', layout=Layout(width='720px'), options=(('All six representative Members', '__al…

Output()

In [80]:
def infer_candidate_field_level(
    stability_table,
):
    """Infer a preliminary candidate storage level."""

    stability_pivot = (
        stability_table
        .pivot(
            index="field",
            columns="level",
            values="stability_fraction",
        )
    )

    availability_pivot = (
        stability_table
        .pivot(
            index="field",
            columns="level",
            values="availability_fraction",
        )
    )

    candidate_rows = []

    for field in stability_pivot.index:
        member_stable = np.isclose(
            stability_pivot.loc[
                field,
                "Member OUS",
            ],
            1.0,
            equal_nan=False,
        )

        source_stable = np.isclose(
            stability_pivot.loc[
                field,
                "Source Context",
            ],
            1.0,
            equal_nan=False,
        )

        spw_stable = np.isclose(
            stability_pivot.loc[
                field,
                "Logical SPW",
            ],
            1.0,
            equal_nan=False,
        )

        record_stable = np.isclose(
            stability_pivot.loc[
                field,
                "Source × SPW Record",
            ],
            1.0,
            equal_nan=False,
        )

        if member_stable:
            candidate_level = "Member OUS"

        elif source_stable and not spw_stable:
            candidate_level = "Source Context"

        elif spw_stable and not source_stable:
            candidate_level = "Logical SPW"

        elif source_stable and spw_stable:
            candidate_level = (
                "Source or SPW — ambiguous"
            )

        elif record_stable:
            candidate_level = (
                "Source × SPW Record"
            )

        else:
            candidate_level = (
                "Raw Archive Row"
            )

        record_availability = (
            availability_pivot.loc[
                field,
                "Source × SPW Record",
            ]
        )

        candidate_rows.append(
            {
                "field": field,
                "preliminary_candidate_level":
                    candidate_level,
                "record_availability":
                    record_availability,
            }
        )

    return pd.DataFrame(
        candidate_rows
    )

In [81]:
preliminary_field_ownership_df = (
    infer_candidate_field_level(
        all_member_stability_df
    )
)


preliminary_field_ownership_df

,field,preliminary_candidate_level,record_availability
0,antenna_arrays,Source Context,1.0
1,asdm_uid,Source Context,1.0
2,bandwidth,Logical SPW,1.0
3,cont_sensitivity_bandwidth,Source Context,1.0
4,frequency,Source × SPW Record,1.0
5,frequency_support,Source Context,1.0
6,group_ous_uid,Member OUS,1.0
7,is_mosaic,Source Context,1.0
8,obs_id,Source × SPW Record,1.0
9,proposal_id,Member OUS,1.0


## Preliminary Internal Data Model v0.1

The following diagram summarizes the preliminary internal data model inferred
from the Archive experiments in Notebooks 1, 2, and 2b.

This is **not a reconstruction of ALMA's complete internal database schema**.
It is a proposed normalized representation for the duplication-checking tool,
derived from:

- official ALMA descriptions of the OUS hierarchy;
- IVOA ObsCore field definitions;
- observed identifier relationships;
- source-by-spectral-window row structures;
- multi-ASDM and mosaic counterexamples;
- field-stability analysis across representative Member OUS datasets.

```mermaid
erDiagram
    PROJECT ||--o{ GROUP_OUS : defines
    PROJECT ||--|{ MEMBER_OUS : includes
    GROUP_OUS o|--o{ MEMBER_OUS : groups

    MEMBER_OUS ||--|{ SOURCE_CONTEXT : contains
    MEMBER_OUS ||--|{ LOGICAL_SPW : defines
    MEMBER_OUS ||--|{ ASDM_EXECUTION : executed_as

    SOURCE_CONTEXT ||--|{ SPATIAL_FOOTPRINT : located_by
    SOURCE_CONTEXT ||--o{ SOURCE_EXEC_CONTEXT : participates_in
    ASDM_EXECUTION ||--o{ SOURCE_EXEC_CONTEXT : provides

    SOURCE_EXEC_CONTEXT ||--|{ SOURCE_SPW_RECORD : produces
    LOGICAL_SPW ||--|{ SOURCE_SPW_RECORD : indexes

    RAW_ARCHIVE_ROW ||--|| SOURCE_SPW_RECORD : maps_to
    PHYSICAL_TARGET o|--o{ SOURCE_CONTEXT : may_unify

    PROJECT {
        string proposal_id PK
    }

    GROUP_OUS {
        string group_ous_uid PK
    }

    MEMBER_OUS {
        string member_ous_uid PK
        string proposal_id FK
        string group_ous_uid FK
    }

    ASDM_EXECUTION {
        string asdm_uid PK
    }

    SOURCE_CONTEXT {
        string source_context_id PK
        string raw_source_label
        string target_name
        string member_ous_uid FK
    }

    SPATIAL_FOOTPRINT {
        string footprint_id PK
        float s_ra_deg
        float s_dec_deg
        string s_region_raw
        string geometry_type
        boolean is_mosaic
    }

    SOURCE_EXEC_CONTEXT {
        string context_id PK
        string source_context_id FK
        string asdm_uid FK
        string antenna_arrays
        float t_min
        float t_max
        float spatial_resolution
        float continuum_sensitivity
        string frequency_support_raw
    }

    LOGICAL_SPW {
        string logical_spw_id PK
        string spw_identifier
        float nominal_bandwidth
    }

    SOURCE_SPW_RECORD {
        string obs_id PK
        float exact_frequency
        float bandwidth
        float line_sensitivity
    }

    RAW_ARCHIVE_ROW {
        string obs_id
        string provenance
        string original_metadata
    }

    PHYSICAL_TARGET {
        string internal_target_id PK
        string normalized_identity
    }
```

### Interpretation

The model separates the flattened Archive response into several conceptual
levels:

1. **Project hierarchy** — Proposal, optional Group OUS, and Member OUS.
2. **Execution provenance** — one Member OUS may reference one or more ASDM
   executions.
3. **Source context** — preserves the raw source label and target metadata
   without assuming that identical or similar names represent the same physical
   target.
4. **Spatial footprint** — stores the reference coordinates and the complete
   `s_region` geometry separately from frequency-related records.
5. **Logical spectral window** — represents the SPW identifier and nominal
   bandwidth.
6. **Source–SPW record** — represents one cell of the observed source-by-SPW
   grid and preserves exact frequency and row-level sensitivity.
7. **Raw Archive row** — preserves the original TAP result for provenance and
   later validation.
8. **Physical target** — a future optional normalization layer that may unify
   multiple source labels without overwriting the original Archive metadata.

The intermediate `SOURCE_EXEC_CONTEXT` entity is included because field
stability alone cannot distinguish source-level properties from
execution-level properties. It allows the same source context to be associated
with multiple ASDM executions, antenna configurations, time ranges, or repeated
observing setups.

### Evidence status

- The OUS hierarchy and the interpretation of Member OUS as an independently
  processable dataset are supported by official ALMA documentation.
- The source-by-SPW grid structure, unique `obs_id` values, variable SPW counts,
  multi-source cases, and multi-ASDM cases are supported by the tested Archive
  samples.
- The placement of some fields, especially `frequency_support`, sensitivity,
  antenna information, and temporal coverage, remains preliminary.
- `PHYSICAL_TARGET` is a proposed application-level entity and is not directly
  supplied by the Archive.
- The model should be revised after frequency-support parsing and current-cycle
  CSV investigation.